# PilotNet V<N> — Analyse des résultats

Notebook réutilisable pour analyser n'importe quelle version (`v1`, `v2`, …).
Modifier la cellule **CONFIGURATION** ci-dessous puis « Run All ».

Sections :

- **A** — Courbes de training (loss, val_loss, par head)
- **B** — Distribution du dataset (steer, speed, throttle/brake, par run)
- **C** — Prédictions vs ground truth sur la val set (scatter, résidus, MAE/RMSE/R²)
- **D** — Bench inférence (ms/frame, FPS atteignable)
- **E** — Run de démo CARLA (timeline vitesse + contrôles, respawns)
- **F** — Dump des metrics consolidés dans `checkpoints/<CHECKPOINT_NAME>/metrics.json`

Génère aussi les figures PNG dans `benchmarks/ai/figures/<MODEL_VERSION>/`.


In [ ]:
# ============================================================================
# CONFIGURATION — modifier ces 4 variables pour analyser une autre version
# ============================================================================
MODEL_VERSION = "v1"                                # → benchmarks/ai/figures/<this>/
CHECKPOINT_NAME = "pilotnet_v1"                     # → checkpoints/<this>/
RUN_GLOB = "2026-05-08_*"                           # → data/runs/<this>
DEMO_LOG_REL = "logs/demo_v1_2026-05-10/demo.log"   # depuis la racine repo
# ============================================================================

import json
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

# Walk up from CWD to find the repo root (works from VSCode or jupyter regardless of CWD)
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Cannot locate project root (no pyproject.toml found)")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CHECKPOINT_DIR = ROOT / "checkpoints" / CHECKPOINT_NAME
FIGURES_DIR = ROOT / "benchmarks" / "ai" / "figures" / MODEL_VERSION
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUN_DIRS = sorted((ROOT / "data" / "runs").glob(RUN_GLOB))
DEMO_LOG = ROOT / DEMO_LOG_REL

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("ROOT:", ROOT)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RUNS:", [r.name for r in RUN_DIRS])
print("DEMO_LOG exists:", DEMO_LOG.exists())


## A — Courbes de training

Chargement de `training_log.csv` et plot des losses globale + par head.

In [ ]:
log = pd.read_csv(CHECKPOINT_DIR / "training_log.csv")
log.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

ax = axes[0, 0]
ax.plot(log["epoch"], log["loss"], label="train", linewidth=2)
ax.plot(log["epoch"], log["val_loss"], label="val", linewidth=2, linestyle="--")
ax.set_title(f"Global weighted loss (final val={log['val_loss'].iloc[-1]:.4f})")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()

for i, head in enumerate(["steer", "throttle", "brake"]):
    row, col = divmod(i + 1, 2)
    ax = axes[row, col]
    ax.plot(log["epoch"], log[f"{head}_loss"], label="train", linewidth=2)
    ax.plot(log["epoch"], log[f"val_{head}_loss"], label="val", linewidth=2, linestyle="--")
    delta = log[f"val_{head}_loss"].iloc[-1] - log[f"val_{head}_loss"].iloc[0]
    sign = "↗" if delta > 0 else "↘"
    ax.set_title(f"{head} MSE — val {sign} ({delta:+.4f})")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE")
    ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## B — Distribution du dataset

Lecture des 3 manifests, filtre `is_collision == 0` (idem data loader), histos steer / speed / throttle / brake et breakdown par run.

In [ ]:
def load_manifest(run_dir):
    df = pd.read_csv(run_dir / "manifest.csv")
    df["run"] = run_dir.name
    return df

df_all = pd.concat([load_manifest(r) for r in RUN_DIRS], ignore_index=True)
df_kept = df_all[df_all["is_collision"] == 0].reset_index(drop=True)

print(f"raw frames: {len(df_all)}")
print(f"kept (no collision): {len(df_kept)}")
print(f"dropped: {len(df_all) - len(df_kept)}")
print()
print("per run:")
print(df_kept["run"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df_kept["steer"], bins=20, range=(-1, 1), edgecolor="black", color="steelblue")

THR = 0.05
n_left = int((df_kept["steer"] < -THR).sum())
n_right = int((df_kept["steer"] > THR).sum())
n_neutral = int((df_kept["steer"].abs() <= THR).sum())
ratio = n_left / max(n_right, 1)

ax.axvline(-THR, color="red", linestyle=":", alpha=0.6)
ax.axvline(THR, color="red", linestyle=":", alpha=0.6)
ax.set_title(
    f"Distribution steer — L={n_left} ({n_left/len(df_kept):.0%}), "
    f"R={n_right} ({n_right/len(df_kept):.0%}), "
    f"neutral={n_neutral} ({n_neutral/len(df_kept):.0%}) — ratio L/R = {ratio:.1f}×"
)
ax.set_xlabel("steer (-1 = left, +1 = right)")
ax.set_ylabel("frames")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "dataset_steer.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.hist(df_kept["speed_kmh"], bins=20, edgecolor="black", color="seagreen")

n_low = int((df_kept["speed_kmh"] < 9).sum())
ax.set_title(
    f"Distribution speed — low (<9 km/h) = {n_low} ({n_low/len(df_kept):.0%}), "
    f"max = {df_kept['speed_kmh'].max():.1f} km/h, mean = {df_kept['speed_kmh'].mean():.1f} km/h"
)
ax.set_xlabel("speed (km/h)")
ax.set_ylabel("frames")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "dataset_speed.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

n_brake_used = int((df_kept["brake"] > 0.1).sum())

axes[0].hist(df_kept["throttle"], bins=20, range=(0, 1), color="darkorange", edgecolor="black")
axes[0].set_title(f"Throttle (mean = {df_kept['throttle'].mean():.2f})")
axes[0].set_xlabel("throttle")
axes[0].set_ylabel("frames")

axes[1].hist(df_kept["brake"], bins=20, range=(0, 1), color="firebrick", edgecolor="black")
axes[1].set_title(f"Brake (mean = {df_kept['brake'].mean():.2f}, frames > 0.1: {n_brake_used})")
axes[1].set_xlabel("brake")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "dataset_throttle_brake.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
per_run = df_kept.groupby("run").agg(
    n=("frame_id", "count"),
    steer_mean=("steer", "mean"),
    speed_mean_kmh=("speed_kmh", "mean"),
    speed_max_kmh=("speed_kmh", "max"),
    pct_left=("steer", lambda s: (s < -0.05).mean()),
    pct_right=("steer", lambda s: (s > 0.05).mean()),
    pct_lowspeed=("speed_kmh", lambda s: (s < 9).mean()),
).round(3)
per_run

## C — Prédictions vs ground truth (val set)

Reconstruction de la val set depuis `splits.json`, inférence du modèle sur les 270 frames, scatter et résidus.

In [ ]:
from src.ai.config import (
    CROP_BOTTOM_PX,
    CROP_TOP_PX,
    IMAGE_HEIGHT,
    IMAGE_RAW_HEIGHT,
    IMAGE_WIDTH,
    SPEED_NORM_DIVISOR,
)

splits = json.loads((CHECKPOINT_DIR / "splits.json").read_text())
val_split = pd.DataFrame(splits["val"], columns=["run_dir_abs", "frame_id"])
val_split["run"] = val_split["run_dir_abs"].apply(lambda p: Path(p).name)
val_df = val_split.merge(df_all, on=["run", "frame_id"], how="left").reset_index(drop=True)
print(f"val rows: {len(val_df)}")
val_df.head()

In [ ]:
def load_image(run_name, image_path):
    p = ROOT / "data" / "runs" / run_name / image_path
    img = tf.io.read_file(str(p))
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.crop_to_bounding_box(
        img,
        CROP_TOP_PX,
        0,
        IMAGE_RAW_HEIGHT - CROP_TOP_PX - CROP_BOTTOM_PX,
        tf.shape(img)[1],
    )
    img = tf.image.resize(img, (IMAGE_HEIGHT, IMAGE_WIDTH))
    return tf.cast(img, tf.float32) / 255.0

print(f"loading {len(val_df)} val images and running inference...")
images = tf.stack([load_image(r["run"], r["image_path"]) for _, r in val_df.iterrows()])
speeds = tf.constant(
    val_df["speed_kmh"].to_numpy().reshape(-1, 1) / SPEED_NORM_DIVISOR,
    dtype=tf.float32,
)
print(f"image tensor: {images.shape}, speed tensor: {speeds.shape}")

model = keras.models.load_model(CHECKPOINT_DIR / "best.keras")
preds = model.predict({"image": images, "speed": speeds}, batch_size=64, verbose=1)

val_df["pred_steer"] = preds["steer"].flatten()
val_df["pred_throttle"] = preds["throttle"].flatten()
val_df["pred_brake"] = preds["brake"].flatten()
val_df[["steer", "pred_steer", "throttle", "pred_throttle", "brake", "pred_brake"]].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

per_head_metrics = {}
for ax, head, lim in zip(axes, ["steer", "throttle", "brake"], [(-1, 1), (0, 1), (0, 1)]):
    true = val_df[head].to_numpy()
    pred = val_df[f"pred_{head}"].to_numpy()
    ax.scatter(true, pred, alpha=0.4, s=14)
    ax.plot(lim, lim, "r--", alpha=0.6, label="y = x")
    ax.set_xlim(*lim)
    ax.set_ylim(*lim)
    ax.set_xlabel(f"true {head}")
    ax.set_ylabel(f"pred {head}")

    mae = float(np.abs(true - pred).mean())
    rmse = float(np.sqrt(((true - pred) ** 2).mean()))
    ss_res = float(((true - pred) ** 2).sum())
    ss_tot = float(((true - true.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / max(ss_tot, 1e-9)

    per_head_metrics[head] = {"mae": mae, "rmse": rmse, "r2": r2}
    ax.set_title(f"{head}\nMAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:+.3f}")
    ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "predictions_scatter.png", dpi=120, bbox_inches="tight")
plt.show()

print(json.dumps(per_head_metrics, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, head in zip(axes, ["steer", "throttle", "brake"]):
    res = (val_df[f"pred_{head}"] - val_df[head]).to_numpy()
    ax.hist(res, bins=40, edgecolor="black", color="slategray")
    ax.axvline(0, color="red", linestyle="--", alpha=0.7)
    ax.set_xlabel(f"residual ({head}: pred - true)")
    ax.set_ylabel("count")
    ax.set_title(f"{head} residuals — mean={res.mean():+.3f}, std={res.std():.3f}")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "predictions_residuals.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# How often does the model emit "active" steering vs the ground truth?
THR = 0.05
true_active = int((val_df["steer"].abs() > THR).sum())
pred_active = int((val_df["pred_steer"].abs() > THR).sum())

# How often does the model emit non-trivial brake?
true_brake = int((val_df["brake"] > 0.1).sum())
pred_brake = int((val_df["pred_brake"] > 0.1).sum())

print(f"Steer 'active' (|val| > {THR}):")
print(f"  ground truth: {true_active}/{len(val_df)} ({true_active/len(val_df):.0%})")
print(f"  model preds : {pred_active}/{len(val_df)} ({pred_active/len(val_df):.0%})")
print()
print(f"Brake > 0.1:")
print(f"  ground truth: {true_brake}/{len(val_df)} ({true_brake/len(val_df):.0%})")
print(f"  model preds : {pred_brake}/{len(val_df)} ({pred_brake/len(val_df):.0%})")

## D — Bench inférence

Mesure du temps de forward sur device courant (CPU ou GPU). Cible CARLA = 20 FPS = budget 50 ms.

In [ ]:
N_WARMUP = 20
N_TIMED = 200

sample_img = images[:1]
sample_speed = speeds[:1]

# Warmup (XLA compilation, kernel autotune, etc.)
for _ in range(N_WARMUP):
    _ = model({"image": sample_img, "speed": sample_speed}, training=False)

# Timed loop
t0 = time.perf_counter()
for _ in range(N_TIMED):
    _ = model({"image": sample_img, "speed": sample_speed}, training=False)
elapsed = time.perf_counter() - t0

ms_per_frame = elapsed / N_TIMED * 1000
fps = 1000.0 / ms_per_frame

device = "GPU" if tf.config.list_physical_devices("GPU") else "CPU"
print(f"Device: {device}")
print(f"Per-frame inference: {ms_per_frame:.2f} ms ({fps:.0f} FPS attainable)")
print(f"CARLA target: 20 FPS = 50 ms budget — {'OK' if ms_per_frame < 50 else 'TOO SLOW'}")

## E — Run de démo CARLA

Parse de `logs/demo_v1_2026-05-10/demo.log` : timeline des collisions, vitesse, contrôles.

In [ ]:
log_text = DEMO_LOG.read_text()

tick_re = re.compile(
    r"\[demo\] t=\s*([\d.]+)s speed=\s*([-\d.]+) "
    r"steer=([+\-\d.]+) throttle=([\d.]+) brake=([\d.]+)"
)
collision_re = re.compile(r"\[demo\] collision #(\d+) at t=([\d.]+)s")

ticks = [
    {
        "t": float(m.group(1)),
        "speed": float(m.group(2)),
        "steer": float(m.group(3)),
        "throttle": float(m.group(4)),
        "brake": float(m.group(5)),
    }
    for m in tick_re.finditer(log_text)
]
collisions = [
    {"n": int(m.group(1)), "t": float(m.group(2))}
    for m in collision_re.finditer(log_text)
]

ticks_df = pd.DataFrame(ticks)
collisions_df = pd.DataFrame(collisions)

print(f"ticks logged: {len(ticks_df)}, collisions: {len(collisions_df)}")
print(f"top speed: {ticks_df['speed'].max():.1f} km/h")
print(f"avg episode duration: {120 / max(len(collisions_df), 1):.1f} s (between collisions)")
print(f"steer mean: {ticks_df['steer'].mean():+.3f} (negative = left bias)")
print(f"model brake usage: {(ticks_df['brake'] > 0.1).mean():.0%} of ticks")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax = axes[0]
ax.plot(ticks_df["t"], ticks_df["speed"], color="steelblue", linewidth=1.6)
for _, c in collisions_df.iterrows():
    ax.axvline(c["t"], color="red", linestyle="--", alpha=0.6)
ax.set_ylabel("speed (km/h)")
ax.set_title(
    f"Demo timeline — {len(collisions_df)} collisions, "
    f"top speed {ticks_df['speed'].max():.1f} km/h, "
    f"avg episode ~{120 / max(len(collisions_df), 1):.0f}s"
)
ax.set_ylim(bottom=0)

ax = axes[1]
ax.plot(ticks_df["t"], ticks_df["steer"], color="darkorange", linewidth=1.2, label="steer")
ax.plot(ticks_df["t"], ticks_df["throttle"], color="seagreen", linewidth=1.2, label="throttle")
ax.plot(ticks_df["t"], ticks_df["brake"], color="firebrick", linewidth=1.2, label="brake")
for _, c in collisions_df.iterrows():
    ax.axvline(c["t"], color="red", linestyle="--", alpha=0.4)
ax.set_xlabel("time (s)")
ax.set_ylabel("control")
ax.legend(loc="upper right")
ax.set_ylim(-1.05, 1.05)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "demo_timeline.png", dpi=120, bbox_inches="tight")
plt.show()

## F — Dump metrics consolidés

Écriture de `checkpoints/pilotnet_v1/metrics.json` pour servir de baseline lors de la comparaison V2.

In [ ]:
metrics = {
    "model_version": MODEL_VERSION,
    "checkpoint": str(CHECKPOINT_DIR / "best.keras"),
    "training": {
        "epochs": int(log["epoch"].max() + 1),
        "loss_final": float(log["loss"].iloc[-1]),
        "val_loss_initial": float(log["val_loss"].iloc[0]),
        "val_loss_final": float(log["val_loss"].iloc[-1]),
        "val_steer_initial": float(log["val_steer_loss"].iloc[0]),
        "val_steer_final": float(log["val_steer_loss"].iloc[-1]),
        "val_throttle_initial": float(log["val_throttle_loss"].iloc[0]),
        "val_throttle_final": float(log["val_throttle_loss"].iloc[-1]),
        "val_brake_initial": float(log["val_brake_loss"].iloc[0]),
        "val_brake_final": float(log["val_brake_loss"].iloc[-1]),
        "early_stopping_triggered": False,
    },
    "dataset": {
        "n_raw": int(len(df_all)),
        "n_kept": int(len(df_kept)),
        "n_dropped_collisions": int(len(df_all) - len(df_kept)),
        "runs": [r.name for r in RUN_DIRS],
        "steer_left_count": n_left,
        "steer_right_count": n_right,
        "steer_neutral_count": n_neutral,
        "steer_left_right_ratio": float(ratio),
        "speed_low_pct": float((df_kept["speed_kmh"] < 9).mean()),
        "speed_max_kmh": float(df_kept["speed_kmh"].max()),
        "speed_mean_kmh": float(df_kept["speed_kmh"].mean()),
        "brake_used_pct": float((df_kept["brake"] > 0.1).mean()),
    },
    "val_predictions": {
        "n_val": int(len(val_df)),
        "per_head": per_head_metrics,
        "steer_active_true_pct": float(true_active / len(val_df)),
        "steer_active_pred_pct": float(pred_active / len(val_df)),
        "brake_active_true_pct": float(true_brake / len(val_df)),
        "brake_active_pred_pct": float(pred_brake / len(val_df)),
    },
    "inference": {
        "device": device,
        "ms_per_frame": ms_per_frame,
        "fps_attainable": fps,
    },
    "demo": {
        "log_path": str(DEMO_LOG.relative_to(ROOT)),
        "duration_s": 120,
        "town": "Town01",
        "weather": "ClearNoon",
        "respawns": int(len(collisions_df)),
        "top_speed_kmh": float(ticks_df["speed"].max()),
        "avg_episode_duration_s": float(120 / max(len(collisions_df), 1)),
        "steer_mean": float(ticks_df["steer"].mean()),
        "model_brake_used_pct": float((ticks_df["brake"] > 0.1).mean()),
    },
}

metrics_path = CHECKPOINT_DIR / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"wrote {metrics_path}")
print()
print(json.dumps(metrics, indent=2))

---

## Bilan / Interprétation

_À remplir après run. Lis les figures et les chiffres ci-dessus, puis remplis chaque section._

### Training

_Trajectoire de val_loss (descente / plateau / divergence), heads qui apprennent vs régressent, EarlyStopping déclenché ou pas._

### Dataset

_Équilibre L/R steer, distribution speed, fréquence du brake actif, biais entre runs._

### Prédictions val

_Forme du nuage scatter par head, mode collapse, écart fréquence d'action prédite vs réelle._

### Démo CARLA

_Nombre de respawns, durée d'épisode médiane, top speed, comportements visibles (biais directionnel, freinage préventif)._

### Diagnostic

_Ce qui marche, ce qui ne marche pas, hypothèses sur les causes._

### Prochaines étapes (V<N+1>)

_Fix prioritaires, expériences à tenter, données à recollecter._
